# 2. Shortcut probe

Can a tiny model tell genuine from forged using **only intensity statistics** (no stroke shape at all)?

Features per image: 32-bin histogram of the median-filtered channel, plus the fraction and mean intensity of
non-white pixels. Logistic regression, 5-fold cross-validation grouped by writer.
If this scores near 100%, the classes are separable by scanner/paper artefacts, and a CNN trained on that input
would learn them instead of forgery cues. Needs the `raw`, `canon` and `binary` caches (`build_cache.ipynb`).

In [1]:
%run ./common.ipynb

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def intensity_features(X):
    ch = X[:, 0].reshape(len(X), -1)            # median-filtered stage
    hist = np.stack([np.histogram(r, bins=32, range=(0, 256))[0] for r in ch]) / ch.shape[1]
    nonwhite = ch < 250
    frac = nonwhite.mean(1, keepdims=True)
    mean_ink = np.array([r[m].mean() if m.any() else 255 for r, m in zip(ch, nonwhite)])[:, None]
    return np.hstack([hist, frac, mean_ink])


meta = pd.read_csv(os.path.join(DATA, "meta.csv"))
y, groups = meta.label.values, meta.writer.values
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
for mode in ["raw", "canon", "binary"]:
    path = os.path.join(DATA, f"cache_{mode}_X.npy")
    if not os.path.exists(path):
        print(f"{mode:7s} no cache (run build_cache.ipynb with this mode)")
        continue
    F = intensity_features(np.load(path, mmap_mode="r"))
    acc = cross_val_score(clf, F, y, groups=groups, cv=GroupKFold(5))
    print(f"{mode:7s} intensity-only accuracy: {acc.mean():.3f} +/- {acc.std():.3f}")

raw     intensity-only accuracy: 1.000 +/- 0.000


canon   intensity-only accuracy: 0.833 +/- 0.039


binary  intensity-only accuracy: 0.559 +/- 0.036
